<a href="https://colab.research.google.com/github/somyadev725-create/fake-job-detector-/blob/main/fake_job_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded = files.upload()

Saving archive.zip to archive.zip


In [3]:
import zipfile
with zipfile.ZipFile("archive.zip") as z:
    z.extractall()
    print(z.namelist())

['fake_job_postings.csv']


In [4]:
import pandas as pd
df = pd.read_csv("fake_job_postings.csv")
print(df.shape)
print(df['fraudulent'].value_counts())
df.head()

(17880, 18)
fraudulent
0    17014
1      866
Name: count, dtype: int64


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


In [5]:
text_cols = ['title', 'company_profile', 'description', 'requirements', 'benefits']
df[text_cols] = df[text_cols].fillna('')
df['text'] = df[text_cols].agg(' '.join, axis=1)

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['fraudulent'],
    test_size=0.2, stratify=df['fraudulent'], random_state=42
)

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=50000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])
model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=50000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [8]:
from sklearn.metrics import classification_report, confusion_matrix
pred = model.predict(X_test)
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

           0       1.00      0.99      0.99      3403
           1       0.81      0.91      0.86       173

    accuracy                           0.99      3576
   macro avg       0.90      0.95      0.93      3576
weighted avg       0.99      0.99      0.99      3576

[[3367   36]
 [  16  157]]


In [9]:
def check_job(text):
    prob = model.predict_proba([text])[0][1]
    return f"Fake probability: {prob:.2%}"

print(check_job("Earn $5000 weekly working from home! No experience needed. Send bank details to start."))

Fake probability: 62.31%


In [10]:
import joblib
from google.colab import files
joblib.dump(model, "fake_job_model.pkl")
files.download("fake_job_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>